In [4]:
from sklearn.datasets import fetch_kddcup99

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_kddcup99

print("Starting KDD Cup 99 clustering experiment (Manual NumPy implementation)...")

# --- 2. Manual Preprocessing Functions ---

def manual_standard_scaler(X):
    """Scales numerical data (X - mean) / std_dev."""
    # Calculate mean and standard deviation along columns
    mean = np.mean(X, axis=0)
    std_dev = np.std(X, axis=0)
    
    # Avoid division by zero for columns with zero variance
    std_dev[std_dev == 0] = 1.0
    
    return (X - mean) / std_dev

def manual_one_hot_encoder(col):
    """Converts a single categorical column into a one-hot encoded array."""
    # Find unique categories and create a mapping
    unique_vals = np.unique(col)
    val_to_index = {val: i for i, val in enumerate(unique_vals)}
    
    # Create the output matrix (n_samples, n_unique_categories)
    n_samples = col.shape[0]
    n_unique = len(unique_vals)
    ohe_matrix = np.zeros((n_samples, n_unique), dtype=int)
    
    # Populate the matrix
    for i, val in enumerate(col):
        ohe_matrix[i, val_to_index[val]] = 1
        
    return ohe_matrix

# --- 3. Manual K-Means Clustering Implementation ---

def manual_euclidean_distances_squared(X, Y):
    """
    Calculates squared Euclidean distances between all pairs of points
    from X (n_samples, n_features) and Y (n_clusters, n_features).
    Returns an (n_samples, n_clusters) matrix.
    """
    # Use broadcasting: (n, 1, d) - (1, k, d) -> (n, k, d)
    diff = X[:, np.newaxis, :] - Y[np.newaxis, :, :]
    # Sum squares along the feature dimension (d)
    return np.sum(diff**2, axis=2)

class ManualKMeans:
    def __init__(self, n_clusters, max_iter=100, random_state=None):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.cluster_centers_ = None
        self.labels_ = None

    def _init_centroids(self, X):
        """Initializes centroids by picking k unique random points from X."""
        n_samples = X.shape[0]
        # Get k unique random indices
        indices = self.rng.choice(n_samples, self.n_clusters, replace=False)
        self.cluster_centers_ = X[indices]

    def fit(self, X):
        """Computes k-means clustering."""
        self.rng = np.random.default_rng(self.random_state)
        self._init_centroids(X)
        
        for _ in range(self.max_iter):
            # Assignment step
            distances = manual_euclidean_distances_squared(X, self.cluster_centers_)
            new_labels = np.argmin(distances, axis=1)
            
            # Update step
            new_centroids = np.zeros_like(self.cluster_centers_)
            empty_clusters = False
            for i in range(self.n_clusters):
                cluster_points = X[new_labels == i]
                if len(cluster_points) > 0:
                    new_centroids[i] = np.mean(cluster_points, axis=0)
                else:
                    # Handle empty cluster: re-initialize its centroid
                    empty_clusters = True
                    new_centroids[i] = X[self.rng.choice(X.shape[0])]

            # Check for convergence
            if not empty_clusters and np.allclose(self.cluster_centers_, new_centroids):
                break
                
            self.cluster_centers_ = new_centroids
            self.labels_ = new_labels
        
        # Final assignment
        distances = manual_euclidean_distances_squared(X, self.cluster_centers_)
        self.labels_ = np.argmin(distances, axis=1)
        return self

    def compute_inertia(self, X):
        """Computes the cost(D, B) function."""
        distances = manual_euclidean_distances_squared(X, self.cluster_centers_)
        # Get the distance for each point to its assigned cluster
        min_distances = distances[np.arange(len(X)), self.labels_]
        return np.sum(min_distances)

# --- 4. Load and Preprocess Data ---

print("Fetching KDD Cup 99 dataset...")
kdd = fetch_kddcup99()
D_raw = kdd.data
n_samples_raw, n_features_raw = D_raw.shape

# Define feature types
categorical_features_idx = [1, 2, 3]
numerical_features_idx = [i for i in range(n_features_raw) 
                          if i not in categorical_features_idx]

print("Preprocessing data (manual scaling and one-hot encoding)...")

# Process numerical features
# Convert to float first, as they are loaded as objects/strings
D_num = D_raw[:, numerical_features_idx].astype(float)
D_num_scaled = manual_standard_scaler(D_num)

# Process categorical features
D_cat_processed_list = []
for i in categorical_features_idx:
    D_cat_col = D_raw[:, i]
    D_cat_ohe = manual_one_hot_encoder(D_cat_col)
    D_cat_processed_list.append(D_cat_ohe)

# Combine all processed features
D = np.hstack([D_num_scaled] + D_cat_processed_list)
n, d = D.shape
print(f"Data preprocessed: n={n} samples, d={d} features (after encoding).")

# --- 5. Define Experiment Parameters ---
k = 15  # Number of clusters
x_values = sorted(list(set([5, 20, 15, 20, 25]))) # Target dimensions
n_runs = 5 # Number of repetitions
results = []

# --- 6. Compute Baseline Cost (cost(D, B)) ---
print(f"\nComputing baseline cost cost(D, B) with k={k} on original data (d={d})...")
kmeans_orig = ManualKMeans(n_clusters=k, random_state=42)
kmeans_orig.fit(D)
cost_D_B = kmeans_orig.compute_inertia(D)
print(f"Baseline cost(D, B): {cost_D_B:.2f}")

# --- 7. Run Experiment Loop ---

for x in x_values:
    print(f"\n--- Running experiments for x = {x} (target dimension) ---")
    for run in range(n_runs):
        run_seed = run  # Use a different seed for each run
        print(f"  Run {run+1}/{n_runs} for x={x} (seed={run_seed})...")
        
        # (a) Define JL matrix M and compute E = DM
        # Create a random Gaussian matrix M (d x x)
        rng_proj = np.random.default_rng(run_seed)
        M = rng_proj.standard_normal((d, x))
        # Project D (n x d) @ M (d x x) -> E (n x x)
        E = D @ M
        
        # (b) Compute k-means on E
        kmeans_proj = ManualKMeans(n_clusters=k, random_state=run_seed)
        kmeans_proj.fit(E)
        proj_labels = kmeans_proj.labels_ 
        
        # (c) Apply "meaningful transformation" to A
        # Find the centroids (A) in the *original* space (R^d)
        # that correspond to the partitioning (proj_labels) found in R^x.
        
        A_recon = np.zeros((k, d))
        for i in range(k):
            points_in_cluster = D[proj_labels == i]
            if points_in_cluster.shape[0] > 0:
                A_recon[i] = np.mean(points_in_cluster, axis=0)
            else:
                # Handle empty cluster (less likely but possible)
                print(f"    Warning: Empty cluster {i} in run {run+1} for x={x}.")
                # Assign a random point from D as centroid
                rng_fallback = np.random.default_rng(run_seed)
                A_recon[i] = D[rng_fallback.choice(n)]

        # (d) Compute cost(D, A)
        # This is the sum of squared distances of each point in D
        # to its cluster's reconstructed centroid (A_recon).
        
        # Calculate all squared distances from D to A_recon
        dist_matrix_DA = manual_euclidean_distances_squared(D, A_recon)
        
        # Select the distance for each point based on its projected label
        cost_D_A = dist_matrix_DA[np.arange(n), proj_labels].sum()
        
        results.append({
            'x': x,
            'run': run + 1,
            'cost(D, A)': cost_D_A,
            'cost(D, B)': cost_D_B
        })
        print(f"    cost(D, A) = {cost_D_A:.2f}")

print("\n--- Experiment Complete ---")

# --- 8. Present Results in a Table ---
results_df = pd.DataFrame(results)
print("\nResults Table (cost(D, A) vs. cost(D, B)):")
print(results_df.to_string())

# Calculate and print averages for summary
avg_costs = results_df.groupby('x')['cost(D, A)'].mean()
print(f"\nBaseline cost(D, B) (Original): {cost_D_B:.2f}")
print("Average cost(D, A) (Projected):")
print(avg_costs)

# --- 9. Present Results in a Bar Graph (Box Plot) ---
print("\nGenerating plot...")
fig, ax = plt.subplots(figsize=(10, 6))

# Create data for the boxplot: a list of arrays, one for each x
boxplot_data = [results_df[results_df['x'] == x_val]['cost(D, A)'].values 
                for x_val in x_values]

ax.boxplot(boxplot_data, labels=x_values, patch_artist=True)


ax.axhline(y=cost_D_B, color='r', linestyle='--', 
           label=f'cost(D, B) (Original Space) = {cost_D_B:.2f}')

ax.set_title('K-Means Cost vs. Random Projection Dimension (k=15, 5 runs)')
ax.set_xlabel('Target Dimension (x)')
ax.set_ylabel('K-Means Cost (Inertia)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

Starting KDD Cup 99 clustering experiment (Manual NumPy implementation)...
Fetching KDD Cup 99 dataset...
Preprocessing data (manual scaling and one-hot encoding)...
Data preprocessed: n=494021 samples, d=118 features (after encoding).

Computing baseline cost cost(D, B) with k=15 on original data (d=118)...
